# Stage 27i TIME_SPEC 100MHz FFT-only Anti-alias Control + Preview

Production-only control for TIME/SPEC science streams and 8-lane DAC-ADC loopback, with RF-reconstructed waveform and FFT-only spectrum preview. Stage 27i keeps 8 TIME flows plus 16 SPEC flows on ports 4300..4323, adds 100MHz anti-alias decim2, and gates on 60Gbps+ combined T510 UDP payload.


In [ ]:
from pathlib import Path
import asyncio
from html import escape
import importlib
import json
import os
import sys
import time
import urllib.request

import ipywidgets as W
import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display

os.environ.setdefault('XILINX_XRT', '/usr')

candidate_roots = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path('/home/xilinx/jupyter_notebooks/t510_fengine'),
    Path('/home/xilinx/t510_fengine_bringup'),
    Path('/home/astrolab/demo-ant'),
]
PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'overlay' / 't510_fengine.bit').exists() and (root / 'python' / 't510_fengine.py').exists():
        PROJECT_ROOT = root
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('overlay/t510_fengine.bit and python/t510_fengine.py were not found')

sys.path.insert(0, str(PROJECT_ROOT))
import python.t510_clock as t510_clock_module
import python.t510_fengine as t510_fengine_module
importlib.reload(t510_clock_module)
importlib.reload(t510_fengine_module)

T510FEngine = t510_fengine_module.T510FEngine
ObservationSpectrumStabilizer = t510_fengine_module.ObservationSpectrumStabilizer

BITFILE = PROJECT_ROOT / 'overlay' / 't510_fengine.bit'
EXPECTED_CORE_VERSION = 0x0001002B
COLORS = ['#0b5cad', '#c45200', '#217a3b', '#b3261e', '#6f42c1', '#5f6368', '#008b8b', '#9a7b00']

state = {
    'core': None,
    'live': False,
    'task': None,
    'last_cfg': None,
    'last_preview': None,
    'last_error': None,
    'last_rates': None,
    'last_rates_time': 0.0,
    'last_xrt': None,
    'stabilizer': ObservationSpectrumStabilizer(),
    'last_stream_ready': None,
}


def set_status(text, *, kind='info'):
    color = {'ok': '#0b7a3b', 'warn': '#9a6a00', 'error': '#b3261e', 'info': '#354052'}.get(kind, '#354052')
    status_html.value = f"<pre style='margin:0;color:{color};white-space:pre-wrap'>{escape(str(text))}</pre>"


def core():
    if state['core'] is None:
        raise RuntimeError('Click Connect first')
    return state['core']


def visible_mask():
    return 0xFF


def phase_values():
    return [float(widget.value) for widget in phase_ch]


def dac_enable_mask():
    return visible_mask()


def capture_input_mask():
    return visible_mask()


def adc_active_mask():
    return T510FEngine.complex_input_mask_to_adc_active_mask(visible_mask())


def amplitude_code():
    return int(round(float(amplitude_percent.value) / 100.0 * 8192.0))


def dac_signal_hz():
    return float(dac_mhz.value) * 1_000_000.0


def science_sample_rate_hz():
    return {20: 30_720_000.0, 100: 122_880_000.0, 200: 245_760_000.0}[int(bandwidth_mhz.value)]


def nearest_fft_bin_alignment():
    fs_hz = science_sample_rate_hz()
    bin_hz = fs_hz / 4096.0
    center_hz = float(center_mhz.value) * 1_000_000.0
    requested_hz = dac_signal_hz()
    signed_bin = int(round((requested_hz - center_hz) / bin_hz))
    aligned_hz = center_hz + signed_bin * bin_hz
    return {
        'sample_rate_hz': fs_hz,
        'bin_width_hz': bin_hz,
        'signed_bin': signed_bin,
        'requested_hz': requested_hz,
        'aligned_hz': aligned_hz,
        'error_hz': requested_hz - aligned_hz,
    }


def format_fft_bin_alignment(info):
    return (
        f"bin={info['signed_bin']} width={info['bin_width_hz']/1_000.0:.3f}kHz "
        f"requested={info['requested_hz']/1_000_000.0:.6f}MHz "
        f"aligned={info['aligned_hz']/1_000_000.0:.6f}MHz "
        f"error={info['error_hz']/1_000.0:+.3f}kHz"
    )


def update_fft_bin_status(info=None):
    if 'fft_bin_status' not in globals():
        return
    info = nearest_fft_bin_alignment() if info is None else info
    fft_bin_status.value = f"<pre style='margin:0'>{escape(format_fft_bin_alignment(info))}</pre>"


def maybe_snap_dac_to_fft_bin():
    info = nearest_fft_bin_alignment()
    info['adjusted'] = False
    if bool(fft_bin_align.value) and abs(float(info['error_hz'])) > 1.0:
        dac_mhz.value = float(info['aligned_hz']) / 1_000_000.0
        info = nearest_fft_bin_alignment()
        info['adjusted'] = True
    update_fft_bin_status(info)
    return info


def science_output_mode_value():
    return str(science_output_mode.value)


def xrt_status_text(status=None):
    status = T510FEngine.ensure_xrt_dri_ready() if status is None else status
    state['last_xrt'] = status
    return (
        f"XRT euid={status.get('euid')} XILINX_XRT={status.get('xilinx_xrt') or '<unset>'} "
        f"zocl={status.get('zocl_loaded')} card0={status.get('dri_card0')} "
        f"renderD128={status.get('dri_renderD128')} by_path={status.get('dri_by_path')}"
    )


def compact_status(status):
    keys = [
        'core_version', 'mode_name', 'science_output_mode', 'sample_rate_hz',
        'time_packet_count', 'spec_packet_count', 'time_dropped_count', 'spec_dropped_count',
        'science_dropped_beat_count', 'tx_route_miss_count', 'tx_route_error_count',
        'tx_frame_dropped_count', 'tx_frame_built_count', 'cmac_link_up', 'cmac_tx_ready',
        'rfdc_active_mask', 'rfdc_current_valid_mask', 'rfdc_seen_valid_mask',
        'pfb_overflow_count', 'pfb_xfft_event_count', 'pfb_xfft_tlast_missing_count',
        'pfb_xfft_tlast_unexpected_count', 'pfb_xfft_fft_overflow_count', 'pfb_xfft_data_out_halt_count',
    ]
    result = {key: status.get(key) for key in keys if key in status}
    aliases = {
        'pfb_overflow_count': 'fft_overflow_count',
        'pfb_xfft_event_count': 'xfft_event_count',
        'pfb_xfft_tlast_missing_count': 'xfft_tlast_missing_count',
        'pfb_xfft_tlast_unexpected_count': 'xfft_tlast_unexpected_count',
        'pfb_xfft_fft_overflow_count': 'xfft_fft_overflow_count',
        'pfb_xfft_data_out_halt_count': 'xfft_data_out_halt_count',
    }
    for old, new in aliases.items():
        if old in result:
            result[new] = result.pop(old)
    return result


def fetch_rust_state():
    url = str(rust_receiver_url.value).strip().rstrip('/')
    if not url:
        return {'skipped': True, 'reason': 'empty Rust receiver URL'}
    try:
        with urllib.request.urlopen(url + '/api/state', timeout=2.0) as response:
            return json.loads(response.read().decode('utf-8'))
    except Exception as exc:
        return {'ok': False, 'url': url, 'error': f'{type(exc).__name__}: {exc}'}


def rust_config_payload():
    return {
        'bandwidth_mhz': int(bandwidth_mhz.value),
        'center_mhz': float(center_mhz.value),
        'expected_mhz': float(dac_mhz.value),
        'dac_mhz': float(dac_mhz.value),
        'waveform_view_mode': str(waveform_view_mode.value),
        'phase_deg_by_channel': phase_values(),
        'channel_mask': visible_mask(),
        'time_window_us': float(time_window_us.value),
        'display_points': int(scope_display_points.value),
        'vertical_scale': float(scope_y_codes.value),
        'paused': False,
    }


def rust_config_matches(config, payload):
    if not isinstance(config, dict):
        return False
    checks = [
        int(config.get('bandwidth_mhz', -1)) == int(payload['bandwidth_mhz']),
        abs(float(config.get('center_mhz', float('nan'))) - float(payload['center_mhz'])) < 1e-6,
        abs(float(config.get('dac_mhz', float('nan'))) - float(payload['dac_mhz'])) < 1e-6,
        str(config.get('waveform_view_mode', 'dual')) == str(payload['waveform_view_mode']),
        abs(float(config.get('time_window_us', float('nan'))) - float(payload['time_window_us'])) < 1e-6,
        int(config.get('display_points', -1)) == int(payload['display_points']),
        int(config.get('channel_mask', -1)) == int(payload['channel_mask']),
        bool(config.get('paused', True)) is False,
    ]
    return all(checks)


def sync_rust_receiver_display():
    url = str(rust_receiver_url.value).strip().rstrip('/')
    if not url:
        return {'skipped': True, 'reason': 'empty Rust receiver URL'}
    payload = rust_config_payload()
    req = urllib.request.Request(
        url + '/api/config',
        data=json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    try:
        with urllib.request.urlopen(req, timeout=2.0) as response:
            applied = json.loads(response.read().decode('utf-8'))
        generation = int(applied.get('config_generation', -1))
        deadline = time.monotonic() + 2.0
        last_state = None
        while time.monotonic() < deadline:
            last_state = fetch_rust_state()
            state_generation = int(last_state.get('config_generation', -1)) if isinstance(last_state, dict) else -1
            if state_generation >= generation and rust_config_matches(last_state.get('config', {}), payload):
                return {'ok': True, 'url': url, 'config_generation': generation, 'confirmed': True}
            time.sleep(0.1)
        return {
            'ok': False,
            'url': url,
            'config_generation': generation,
            'confirmed': False,
            'error': 'Rust receiver did not confirm applied display config within 2s',
            'last_state_generation': None if not isinstance(last_state, dict) else last_state.get('config_generation'),
        }
    except Exception as exc:
        return {'ok': False, 'url': url, 'error': f'{type(exc).__name__}: {exc}'}


def get_rates(c, *, force=False):
    now = time.monotonic()
    if not force and state['last_rates'] is not None and now - state['last_rates_time'] < 0.8:
        return state['last_rates']
    rates = {
        'status': c.read_status(),
        'science': c.read_science_output_status(),
        'tx': c.read_tx_status(),
        'channelizer': c.read_channelizer_status(),
        'time_routes': c.read_time_route_table(),
        'spec_routes': c.read_spec_route_table(),
        'rust': fetch_rust_state(),
    }
    state['last_rates'] = rates
    state['last_rates_time'] = now
    return rates


def route_hit_count(routes):
    return sum(1 for route in routes if int(route.get('enable', 0)) and int(route.get('hit_count', 0)) > 0)


def mode_requires_time(mode=None):
    mode = science_output_mode_value() if mode is None else str(mode)
    return mode in ('time_only', 'time_spec')


def mode_requires_spec(mode=None):
    mode = science_output_mode_value() if mode is None else str(mode)
    return mode in ('spec_only', 'time_spec')


def _count(status, key):
    return int(status.get(key, 0) or 0)


def stream_ready_snapshot(c, *, before=None, require_packet_advance=True):
    status = c.read_status()
    tx = c.read_tx_status()
    science = c.read_science_output_status()
    required_adc = int(adc_active_mask())
    valid_mask = int(status.get('rfdc_current_valid_mask', 0))
    streaming = bool(int(status.get('streaming', 0)))
    rfdc_valid = (valid_mask & required_adc) == required_adc
    tx_ready = bool(int(tx.get('cmac_tx_ready', status.get('cmac_tx_ready', 0)) or 0))
    mode = science_output_mode_value()
    before = before or {}
    time_packets = _count(status, 'time_packet_count')
    spec_packets = _count(status, 'spec_packet_count')
    time_ok = (not mode_requires_time(mode)) or (not require_packet_advance) or time_packets > int(before.get('time_packet_count', -1))
    spec_ok = (not mode_requires_spec(mode)) or (not require_packet_advance) or spec_packets > int(before.get('spec_packet_count', -1))
    ok = streaming and rfdc_valid and tx_ready and time_ok and spec_ok
    return {
        'ok': ok,
        'mode': mode,
        'streaming': streaming,
        'rfdc_valid': rfdc_valid,
        'required_adc_mask': required_adc,
        'rfdc_current_valid_mask': valid_mask,
        'tx_ready': tx_ready,
        'time_packets': time_packets,
        'spec_packets': spec_packets,
        'time_advanced': time_ok,
        'spec_advanced': spec_ok,
        'science_output_mode': science.get('science_output_mode'),
    }


def wait_production_stream_ready(c, *, before=None, timeout_s=4.0, require_packet_advance=True):
    deadline = time.monotonic() + float(timeout_s)
    last = None
    while time.monotonic() < deadline:
        last = stream_ready_snapshot(c, before=before, require_packet_advance=require_packet_advance)
        if last['ok']:
            state['last_stream_ready'] = last
            return last
        set_status('Stage 27h stream starting\n' + json.dumps(last, indent=2, sort_keys=True), kind='info')
        time.sleep(0.1)
    state['last_stream_ready'] = last
    raise TimeoutError('Stage 27h stream did not reach production-ready state: ' + json.dumps(last, sort_keys=True))


def ensure_preview_streaming(c):
    status = c.read_status()
    if int(status.get('streaming', 0)) and (int(status.get('rfdc_current_valid_mask', 0)) & int(adc_active_mask())):
        return {'ok': True, 'already_streaming': True}
    return wait_production_stream_ready(c, timeout_s=2.0, require_packet_advance=False)


def update_status_panel(status=None, analysis=None, rates=None):
    c = state.get('core')
    if c is None:
        board_html.value = '<pre style="margin:0">Board not connected.</pre>'
        rust_html.value = '<pre style="margin:0">Rust receiver not queried.</pre>'
        preview_html.value = '<pre style="margin:0">Preview not captured.</pre>'
        return
    rates = get_rates(c, force=True) if rates is None else rates
    status = rates['status'] if status is None else status
    science = rates.get('science', {})
    tx = rates.get('tx', {})
    channelizer = rates.get('channelizer', {})
    time_routes = rates.get('time_routes', [])
    spec_routes = rates.get('spec_routes', [])
    active_time = sum(1 for route in time_routes if int(route.get('enable', 0)))
    active_spec = sum(1 for route in spec_routes if int(route.get('enable', 0)))
    board_html.value = '<pre style="margin:0;white-space:pre-wrap">' + '\n'.join([
        f"core=0x{int(status.get('core_version', 0)):08x} mode={status.get('mode_name', status.get('mode'))} science={science.get('science_output_mode')}",
        f"TIME packets={int(status.get('time_packet_count', 0))} drops={int(status.get('time_dropped_count', 0))} bytes={int(status.get('time_udp_byte_count', 0))} routes={route_hit_count(time_routes)}/{active_time or 8}",
        f"SPEC packets={int(status.get('spec_packet_count', 0))} drops={int(status.get('spec_dropped_count', 0))} bytes={int(status.get('spec_udp_byte_count', 0))} routes={route_hit_count(spec_routes)}/{active_spec or 16}",
        f"F-engine FFT-only={channelizer.get('pfb_fft_only')} taps={channelizer.get('pfb_taps')} chan_count={channelizer.get('pfb_chan_count')} time_count={channelizer.get('pfb_time_count')}",
        f"F-engine overflow={int(status.get('pfb_overflow_count', 0))} xfft_event={int(status.get('pfb_xfft_event_count', 0))} science_drop={int(status.get('science_dropped_beat_count', 0))}",
        f"TX frames={int(status.get('tx_frame_built_count', 0))} route_miss={int(status.get('tx_route_miss_count', 0))} route_error={int(status.get('tx_route_error_count', 0))} frame_drop={int(status.get('tx_frame_dropped_count', 0))}",
        f"CMAC link={tx.get('link_up', status.get('cmac_link_up'))} ready={tx.get('cmac_tx_ready', status.get('cmac_tx_ready'))} local_fault={tx.get('tx_local_fault')} remote_fault={tx.get('tx_remote_fault')}",
        f"RFDC active=0x{int(status.get('rfdc_active_mask', 0)):04x} valid=0x{int(status.get('rfdc_current_valid_mask', 0)):04x} seen=0x{int(status.get('rfdc_seen_valid_mask', 0)):04x}",
        xrt_status_text(state.get('last_xrt')),
    ]) + '</pre>'

    rust = rates.get('rust', {})
    if isinstance(rust, dict) and 'stats' in rust:
        s = rust['stats']
        rust_html.value = '<pre style="margin:0;white-space:pre-wrap">' + '\n'.join([
            f"url={str(rust_receiver_url.value).strip().rstrip('/')} backend={s.get('backend')} fanout={s.get('fanout_mode')} workers={s.get('active_worker_count')}/{s.get('worker_count')}",
            f"bandwidth selected={s.get('selected_bandwidth_mhz')}MHz detected={s.get('detected_bandwidth_mhz')}MHz mismatch={s.get('selected_detected_mismatch')}",
            f"TIME packets={s.get('time_packets')} processed={float(s.get('rx_processed_gbps', 0.0)):.3f}Gbps gaps={s.get('seq_gaps')}/{s.get('frame_gaps')}/{s.get('sample0_gaps')}",
            f"SPEC packets={s.get('spec_packets')} processed={float(s.get('spec_processed_gbps', 0.0)):.3f}Gbps gaps={s.get('spec_seq_gaps')}/{s.get('spec_frame_gaps')}",
            f"drops parse={s.get('parse_errors')} ring={s.get('ring_drops')} worker_ring={s.get('worker_ring_drops')} kernel={s.get('kernel_drops')}",
            f"preview waveform={float(s.get('display_update_hz', 0.0)):.2f}Hz spectrum={float(s.get('spectrum_update_hz', 0.0)):.2f}Hz ws={s.get('websocket_clients')}",
        ]) + '</pre>'
    else:
        rust_html.value = '<pre style="margin:0;white-space:pre-wrap">' + escape(json.dumps(rust, indent=2, sort_keys=True, default=str)) + '</pre>'

    if analysis is not None:
        peaks = analysis.get('peaks', {})
        measured = analysis.get('measured_peak', {}) or {}
        lines = [
            f"sample_rate={float(analysis.get('sample_rate_hz', 0.0))/1e6:.3f}MS/s requested={float(analysis.get('requested_window_us', analysis.get('time_window_us', 0.0))):.3f}us captured={float(analysis.get('captured_window_us', 0.0)):.3f}us real={int(analysis.get('display_count', 0))}/{int(analysis.get('count', 0))} curve={int(analysis.get('rf_curve_point_count', 0))}",
            f"expected RF={float(analysis.get('expected_rf_hz', 0.0))/1e6:.4f}MHz baseband={float(analysis.get('expected_baseband_hz', 0.0))/1e6:.4f}MHz measured RF={float(measured.get('rf_peak_mhz', 0.0)):.4f}MHz baseband={float(measured.get('baseband_mhz', 0.0)):.4f}MHz",
            f"RF samples/cycle={float(analysis.get('rf_samples_per_cycle', float('inf'))):.2f} baseband samples/cycle={float(analysis.get('baseband_samples_per_cycle', float('inf'))):.2f} RF cycles/window={float(analysis.get('expected_cycles_in_window', 0.0)):.2f}",
            f"provenance: measured IQ from RFDC preview, RF-equivalent derived_from_real_iq=True raw_rf=False",
        ]
        if analysis.get('rf_near_nyquist_warning'):
            lines.append('WARNING: expected RF is near Nyquist in preview; validate sine shape with baseband IQ and spectrum, not RF connected-line shape.')
        ref_phase = None
        for ch in range(8):
            if ch in peaks:
                pk = peaks[ch]
                if ref_phase is None:
                    ref_phase = float(pk.get('coherent_phase_deg', 0.0))
                lines.append(
                    f"CH{ch} RF={float(pk.get('rf_peak_mhz', 0.0)):.4f}MHz baseband={float(pk.get('baseband_mhz', 0.0)):.4f}MHz "
                    f"peak={float(pk.get('peak_dbfs', 0.0)):.2f}dBFS SNR={float(pk.get('snr_db', 0.0)):.1f}dB "
                    f"phase={float(pk.get('coherent_phase_deg', 0.0)):.1f}deg rel={float(pk.get('delta_coherent_phase_deg', 0.0)):.1f}deg"
                )
        preview_html.value = '<pre style="margin:0;white-space:pre-wrap">' + escape('\n'.join(lines)) + '</pre>'


def connect_core(*, download=False):
    xrt = T510FEngine.ensure_xrt_dri_ready()
    state['last_xrt'] = xrt
    state['core'] = T510FEngine(str(BITFILE), download=bool(download))
    status = state['core'].read_status()
    core_version = int(status.get('core_version', 0))
    action = 'Reprogrammed PL and connected' if download else 'Connected to current PL image'
    kind = 'ok' if core_version == EXPECTED_CORE_VERSION else 'warn'
    set_status(
        f"{action}: {BITFILE}\ncore_version=0x{core_version:08x} expected=0x{EXPECTED_CORE_VERSION:08x}\n" +
        xrt_status_text(xrt) + '\n' +
        json.dumps(compact_status(status), indent=2, sort_keys=True),
        kind=kind,
    )
    update_status_panel(status=status)


def connect_clicked(_=None):
    try:
        connect_core(download=bool(reprogram_pl.value))
    except Exception as exc:
        state['last_error'] = exc
        set_status(
            f"Connect failed: {type(exc).__name__}: {exc}\n{xrt_status_text(state.get('last_xrt'))}",
            kind='error',
        )


def reload_clicked(_=None):
    try:
        connect_core(download=True)
    except Exception as exc:
        state['last_error'] = exc
        set_status(
            f"Reload failed: {type(exc).__name__}: {exc}\n{xrt_status_text(state.get('last_xrt'))}",
            kind='error',
        )


def apply_production_config(*, initialize=False, start=True):
    c = core()
    if science_output_mode_value() == 'time_spec' and int(bandwidth_mhz.value) == 200:
        raise ValueError('Stage 27h FFT-only production rejects TIME_SPEC at 200 MHz')
    alignment = maybe_snap_dac_to_fft_bin()
    set_status('Applying Stage 27h production config and starting stream...\n' + format_fft_bin_alignment(alignment), kind='info')
    observation = c.apply_mts_locked_observation_config(
        observe_center_hz=float(center_mhz.value) * 1_000_000.0,
        dac_signal_hz=dac_signal_hz(),
        expected_signal_hz=dac_signal_hz(),
        view_bw_hz=float(bandwidth_mhz.value) * 1_000_000.0,
        amplitude=amplitude_code(),
        phase_deg=0.0,
        phase_deg_by_channel=phase_values(),
        enable_mask=dac_enable_mask(),
        adc_active_mask=adc_active_mask(),
        initialize=bool(initialize),
        start=False,
        require_full_clock_lock=True,
        require_mts=True,
        force_clock_reconfigure=bool(initialize),
        input_source_mode='dac_loopback',
        clock_ref='external_10mhz',
        sync_mode='external_pps',
    )
    science = c.configure_science_27h(
        bandwidth_mhz=int(bandwidth_mhz.value),
        output_mode=science_output_mode_value(),
        dst_ip=str(dst_ip.value).strip(),
        dst_mac=str(dst_mac.value).strip(),
        src_ip=str(src_ip.value).strip(),
        src_mac=str(src_mac.value).strip(),
        time_dst_port_base=int(time_dst_port_base.value),
        spec_dst_port_base=int(spec_dst_port_base.value),
        time_src_port_base=int(time_src_port_base.value),
        spec_src_port_base=int(spec_src_port_base.value),
        input_mask=0x00FF,
        spec_route_count=16,
        spec_chan_count=256,
        spec_time_count=1,
        spec_chan0_stride=256,
        pfb_taps=0,
        clear_counters=True,
        start=bool(start),
    )
    before = c.read_status()
    rust_sync = sync_rust_receiver_display()
    stream_ready = None
    if start:
        stream_ready = wait_production_stream_ready(
            c,
            before={
                'time_packet_count': int(before.get('time_packet_count', 0)),
                'spec_packet_count': int(before.get('spec_packet_count', 0)),
            },
            timeout_s=5.0,
            require_packet_advance=True,
        )
    cfg = {'observation': observation, 'science': science, 'rust_receiver_sync': rust_sync, 'stream_ready': stream_ready, 'fft_bin_alignment': alignment}
    state['last_cfg'] = cfg
    state['last_rates'] = None
    update_status_panel()
    return cfg


def init_clicked(_=None):
    try:
        cfg = apply_production_config(initialize=True, start=True)
        set_status('Initialized RFDC/DAC loopback and verified Stage 27h stream ready\n' + json.dumps({'science': cfg['science'], 'rust_receiver_sync': cfg['rust_receiver_sync'], 'stream_ready': cfg['stream_ready']}, indent=2, sort_keys=True, default=str), kind='ok')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Init failed: {type(exc).__name__}: {exc}', kind='error')


def apply_clicked(_=None):
    try:
        cfg = apply_production_config(initialize=False, start=True)
        set_status('Applied Stage 27h production config and verified stream ready\n' + json.dumps({'science': cfg['science'], 'rust_receiver_sync': cfg['rust_receiver_sync'], 'stream_ready': cfg['stream_ready']}, indent=2, sort_keys=True, default=str), kind='ok')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Apply failed: {type(exc).__name__}: {exc}', kind='error')


def apply_phase_only(_=None):
    try:
        c = core()
        c.configure_dac_tone_bank(
            freq_hz=0.0,
            amplitude=amplitude_code(),
            phase_offset_deg=0.0,
            phase_deg_by_channel=phase_values(),
            enable_mask=dac_enable_mask(),
            mode='constant_phasor',
        )
        epoch = c.reset_dac_phase()
        sync_rust_receiver_display()
        set_status(f'Applied DAC constant-phasor amplitude and per-lane phases epoch={epoch}. Use Apply for RF frequency/NCO changes.', kind='ok')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Phase apply failed: {type(exc).__name__}: {exc}', kind='error')


def stop_clicked(_=None):
    try:
        stop_live()
        core().stop()
        update_status_panel()
        set_status('Stopped production stream and live preview.', kind='warn')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Stop failed: {type(exc).__name__}: {exc}', kind='error')


def reduce_xy(x, y, max_points):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.size <= int(max_points):
        return x, y
    idx = np.linspace(0, x.size - 1, int(max_points)).astype(int)
    return x[idx], y[idx]


def capture_count():
    return T510FEngine.observation_capture_count(
        sample_rate_hz=245_760_000.0,
        time_window_us=float(time_window_us.value),
        oversample=float(fft_padding_factor.value),
        min_count=512,
        max_count=int(preview_max_samples.value),
    )


def update_waveform_plot(analysis):
    y_limit = float(scope_y_codes.value)
    max_points = int(scope_display_points.value)
    mode = str(waveform_view_mode.value)
    with rf_fig.batch_update():
        for ch in range(8):
            curve_trace = rf_fig.data[ch * 2]
            sample_trace = rf_fig.data[ch * 2 + 1]
            if ch not in analysis['scope']:
                curve_trace.visible = False
                sample_trace.visible = False
                continue
            scope = analysis['scope'][ch]
            curve_x, curve_y = reduce_xy(scope.get('rf_equivalent_curve_time_us', scope['rf_equivalent_time_us']), scope.get('rf_equivalent_curve_waveform', scope['rf_equivalent_waveform']), max_points)
            sample_x = np.asarray(scope['rf_equivalent_time_us'], dtype=float)
            sample_y = np.asarray(scope['rf_equivalent_waveform'], dtype=float)
            curve_trace.x = curve_x
            curve_trace.y = curve_y
            curve_trace.mode = 'lines'
            curve_trace.visible = mode in ('dual', 'curve')
            curve_trace.name = f'CH{ch} RF curve'
            sample_trace.x = sample_x
            sample_trace.y = sample_y
            sample_trace.mode = 'markers'
            sample_trace.visible = mode in ('dual', 'samples')
            sample_trace.name = f'CH{ch} real samples'
        requested = float(analysis.get('requested_window_us', analysis.get('time_window_us', 0.0)))
        captured = float(analysis.get('captured_window_us', 0.0))
        cycles = float(analysis.get('expected_cycles_in_window', 0.0))
        rf_fig.layout.title = f'Production RF-equivalent waveform {mode}: requested={requested:.3f}us captured={captured:.3f}us cycles={cycles:.2f}'
        rf_fig.layout.xaxis.range = [0.0, requested]
        rf_fig.layout.yaxis.range = [-y_limit, y_limit]


def update_spectrum_plot(analysis):
    xmin = float(center_mhz.value) - float(bandwidth_mhz.value) / 2.0
    xmax = float(center_mhz.value) + float(bandwidth_mhz.value) / 2.0
    with spectrum_fig.batch_update():
        for ch in range(8):
            trace = spectrum_fig.data[ch]
            if ch not in analysis['spectrum']:
                trace.visible = False
                continue
            spectrum = analysis['spectrum'][ch]
            peak = analysis['peaks'].get(ch, {})
            update = state['stabilizer'].update_channel(
                ch,
                spectrum,
                peak,
                smoothing_enabled=bool(smoothing_enabled.value),
                alpha=float(smoothing_alpha.value),
            )
            x, y = reduce_xy(update['rf_mhz'], update['display_power_dbfs'], int(spectrum_display_points.value))
            trace.x = x
            trace.y = y
            trace.visible = True
            trace.name = f'CH{ch}'
        spectrum_fig.layout.xaxis.range = [xmin, xmax]
        spectrum_fig.layout.yaxis.range = [int(spectrum_floor_db.value), int(spectrum_top_db.value)]
        spectrum_fig.layout.title = f'FFT-only production RF spectrum center={float(center_mhz.value):.1f} MHz BW={float(bandwidth_mhz.value):.0f} MHz'


def capture_once(_=None):
    try:
        c = core()
        t0 = time.monotonic()
        ensure_preview_streaming(c)
        preview = c.capture_preview_fast(n=capture_count(), input_mask=capture_input_mask(), timeout=float(timeout_s.value))
        state['last_preview'] = preview
        analysis = c.compute_observation_view(
            preview,
            observe_center_hz=float(center_mhz.value) * 1_000_000.0,
            view_bw_hz=float(bandwidth_mhz.value) * 1_000_000.0,
            dac_signal_hz=dac_signal_hz(),
            expected_signal_hz=dac_signal_hz(),
            time_window_us=float(time_window_us.value),
            curve_points=int(scope_display_points.value),
            oversample=float(fft_padding_factor.value),
            phase_ref_input=0,
            stabilize_phase=False,
            display_phase_deg=0.0,
            phase_deg_by_channel=phase_values(),
            input_source_mode='dac_loopback',
        )
        update_waveform_plot(analysis)
        update_spectrum_plot(analysis)
        rates = get_rates(c, force=True)
        elapsed = time.monotonic() - t0
        update_status_panel(analysis=analysis, rates=rates)
        set_status(f"Captured production preview sample0={int(preview['sample0'])} count={int(preview['count'])} elapsed={elapsed*1000.0:.1f}ms", kind='ok')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Capture failed: {type(exc).__name__}: {exc}', kind='error')


async def live_loop():
    while state.get('live', False):
        capture_once()
        await asyncio.sleep(max(1.0 / max(float(refresh_hz.value), 0.1), float(min_ui_sleep_ms.value) / 1000.0))


def start_live(_=None):
    try:
        if state.get('live', False):
            return
        state['live'] = True
        state['task'] = asyncio.create_task(live_loop())
        set_status('Live production preview started.', kind='ok')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Start live failed: {type(exc).__name__}: {exc}', kind='error')


def stop_live(_=None):
    state['live'] = False
    task = state.get('task')
    if task is not None:
        task.cancel()
    state['task'] = None
    set_status('Live preview stopped.', kind='warn')


def validate_board(_=None):
    try:
        result = core().run_stage27h_time_spec_fft_fullrate_validation(
            configure=False,
            expected_core_version=EXPECTED_CORE_VERSION,
            bandwidth_mhz=int(bandwidth_mhz.value),
            output_mode=science_output_mode_value(),
            seconds=float(validation_seconds.value),
            min_time_pps=470_000.0,
            min_spec_pps=470_000.0,
            min_combined_t510_udp_payload_mbps=63_000.0,
        )
        update_status_panel()
        set_status(json.dumps(result, indent=2, sort_keys=True, default=str), kind='ok' if result.get('ok') else 'error')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Board validation failed: {type(exc).__name__}: {exc}', kind='error')


def refresh_status(_=None):
    try:
        update_status_panel()
        set_status('Status refreshed. ' + xrt_status_text(state.get('last_xrt')), kind='info')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Status refresh failed: {type(exc).__name__}: {exc}', kind='error')


def on_mode_or_bw_change(change=None):
    if science_output_mode_value() == 'time_spec' and int(bandwidth_mhz.value) == 200:
        set_status('TIME_SPEC 200MHz is rejected by Stage 27h FFT-only production; choose 100MHz or change mode.', kind='warn')
    update_fft_bin_status()


def on_fft_bin_control_change(change=None):
    update_fft_bin_status()


def align_dac_clicked(_=None):
    try:
        info = maybe_snap_dac_to_fft_bin()
        set_status('Snapped DAC RF to nearest FFT bin.\n' + format_fft_bin_alignment(info), kind='ok')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'FFT-bin snap failed: {type(exc).__name__}: {exc}', kind='error')


style = {'description_width': '130px'}
wide = W.Layout(width='430px')
reprogram_pl = W.Checkbox(value=False, description='Reprogram PL')
rust_receiver_url = W.Text(value='http://192.168.100.192:8089', description='Rust receiver', style=style, layout=wide)
dst_ip = W.Text(value='10.0.1.16', description='Receiver IP', style=style, layout=wide)
dst_mac = W.Text(value='08:c0:eb:d5:95:b2', description='Receiver MAC', style=style, layout=wide)
src_ip = W.Text(value='10.0.1.1', description='Source IP', style=style, layout=wide)
src_mac = W.Text(value='02:00:00:00:00:01', description='Source MAC', style=style, layout=wide)
time_dst_port_base = W.IntText(value=4300, description='TIME dst port', style=style, layout=wide)
spec_dst_port_base = W.IntText(value=4308, description='SPEC dst port', style=style, layout=wide)
time_src_port_base = W.IntText(value=4000, description='TIME src port', style=style, layout=wide)
spec_src_port_base = W.IntText(value=4008, description='SPEC src port', style=style, layout=wide)
science_output_mode = W.Dropdown(options=[('TIME + SPEC', 'time_spec'), ('TIME only', 'time_only'), ('SPEC only', 'spec_only')], value='time_spec', description='Mode', style=style, layout=wide)
bandwidth_mhz = W.Dropdown(options=[('100 MHz', 100), ('20 MHz', 20), ('200 MHz', 200)], value=100, description='Bandwidth', style=style, layout=wide)
center_mhz = W.FloatText(value=100.0, description='Center MHz', style=style, layout=wide)
dac_mhz = W.FloatText(value=60.010, description='DAC RF MHz', style=style, layout=wide)
fft_bin_align = W.Checkbox(value=True, description='Snap DAC to FFT bin')
fft_bin_status = W.HTML(value='<pre style="margin:0">FFT bin status pending.</pre>')
amplitude_percent = W.FloatSlider(value=25.0, min=0.0, max=100.0, step=1.0, description='DAC amplitude %', readout_format='.0f', style=style, layout=wide)
validation_seconds = W.FloatSlider(value=2.0, min=0.5, max=30.0, step=0.5, description='Validate s', readout_format='.1f', style=style, layout=wide)
phase_ch = [W.FloatSlider(value=0.0, min=-180.0, max=180.0, step=1.0, description=f'CH{i}', readout_format='.0f', continuous_update=False, style={'description_width': '48px'}, layout=W.Layout(width='260px')) for i in range(8)]
scope_y_codes = W.IntSlider(value=4096, min=256, max=32768, step=256, description='Y scale codes', style=style, layout=wide)
scope_display_points = W.IntSlider(value=1024, min=256, max=4096, step=256, description='Display points', style=style, layout=wide)
waveform_view_mode = W.Dropdown(options=[('Curve + samples', 'dual'), ('Samples only', 'samples'), ('Curve only', 'curve')], value='dual', description='Waveform mode', style=style, layout=wide)
time_window_us = W.FloatSlider(value=0.25, min=0.05, max=2.0, step=0.05, description='Window us', readout_format='.2f', style=style, layout=wide)
fft_padding_factor = W.FloatSlider(value=2.5, min=1.0, max=8.0, step=0.5, description='FFT padding factor', style=style, layout=wide)
preview_max_samples = W.Dropdown(options=[512, 1024, 2048, 4096], value=1024, description='Max samples', style=style, layout=wide)
refresh_hz = W.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='Preview Hz', readout_format='.1f', style=style, layout=wide)
min_ui_sleep_ms = W.IntSlider(value=30, min=0, max=250, step=10, description='UI sleep ms', style=style, layout=wide)
timeout_s = W.FloatSlider(value=1.0, min=0.2, max=5.0, step=0.2, description='Timeout s', readout_format='.1f', style=style, layout=wide)
spectrum_floor_db = W.IntSlider(value=-120, min=-160, max=-20, step=5, description='Spectrum min', style=style, layout=wide)
spectrum_top_db = W.IntSlider(value=-20, min=-80, max=10, step=5, description='Spectrum max', style=style, layout=wide)
spectrum_display_points = W.IntSlider(value=512, min=128, max=2048, step=128, description='Spectrum points', style=style, layout=wide)
smoothing_enabled = W.Checkbox(value=True, description='Smooth spectrum')
smoothing_alpha = W.FloatSlider(value=0.25, min=0.05, max=1.0, step=0.05, description='Smooth alpha', style=style, layout=wide)

connect_btn = W.Button(description='Connect', icon='plug', button_style='primary', layout=W.Layout(width='105px'))
reload_btn = W.Button(description='Reload bit', icon='download', layout=W.Layout(width='110px'))
init_btn = W.Button(description='Init + Start', icon='play', button_style='success', layout=W.Layout(width='125px'))
apply_btn = W.Button(description='Apply', icon='check', button_style='success', layout=W.Layout(width='90px'))
phase_btn = W.Button(description='Apply phase', icon='sliders', layout=W.Layout(width='125px'))
align_dac_btn = W.Button(description='Snap bin', icon='crosshairs', layout=W.Layout(width='105px'))
capture_btn = W.Button(description='Capture', icon='camera', layout=W.Layout(width='100px'))
start_live_btn = W.Button(description='Live preview', icon='play', layout=W.Layout(width='125px'))
stop_live_btn = W.Button(description='Stop live', icon='stop', button_style='warning', layout=W.Layout(width='105px'))
validate_btn = W.Button(description='Board gate', icon='shield-check', button_style='warning', layout=W.Layout(width='115px'))
status_btn = W.Button(description='Status', icon='activity', layout=W.Layout(width='90px'))
stop_btn = W.Button(description='Stop stream', icon='square', button_style='warning', layout=W.Layout(width='115px'))

status_html = W.HTML(value='<pre style="margin:0">Connect to the current Stage 27h overlay to begin.</pre>')
board_html = W.HTML(value='<pre style="margin:0">Board not connected.</pre>')
rust_html = W.HTML(value='<pre style="margin:0">Rust receiver not queried.</pre>')
preview_html = W.HTML(value='<pre style="margin:0">Capture a frame to populate preview metrics.</pre>')

rf_fig = go.FigureWidget()
for ch in range(8):
    rf_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name=f'CH{ch} RF curve', line={'color': COLORS[ch], 'width': 2 if ch == 0 else 1.2}, visible=True))
    rf_fig.add_trace(go.Scattergl(x=[], y=[], mode='markers', name=f'CH{ch} real samples', marker={'color': COLORS[ch], 'size': 4}, visible=True))
rf_fig.update_layout(height=360, margin={'l': 58, 'r': 20, 't': 38, 'b': 46}, xaxis_title='preview time (us)', yaxis_title='ADC code', template='plotly_white', showlegend=True, title='Production RF-equivalent waveform from real RFDC IQ samples')

spectrum_fig = go.FigureWidget()
for ch in range(8):
    spectrum_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name=f'CH{ch}', line={'color': COLORS[ch], 'width': 2 if ch == 0 else 1.2}, visible=(ch == 0)))
spectrum_fig.update_layout(height=360, margin={'l': 58, 'r': 20, 't': 38, 'b': 46}, xaxis_title='RF frequency (MHz)', yaxis_title='power (dBFS)', template='plotly_white', showlegend=True, title='FFT-only production RF spectrum')

connect_btn.on_click(connect_clicked)
reload_btn.on_click(reload_clicked)
init_btn.on_click(init_clicked)
apply_btn.on_click(apply_clicked)
phase_btn.on_click(apply_phase_only)
align_dac_btn.on_click(align_dac_clicked)
capture_btn.on_click(capture_once)
start_live_btn.on_click(start_live)
stop_live_btn.on_click(stop_live)
validate_btn.on_click(validate_board)
status_btn.on_click(refresh_status)
stop_btn.on_click(stop_clicked)
science_output_mode.observe(on_mode_or_bw_change, names='value')
bandwidth_mhz.observe(on_mode_or_bw_change, names='value')
center_mhz.observe(on_fft_bin_control_change, names='value')
dac_mhz.observe(on_fft_bin_control_change, names='value')
fft_bin_align.observe(on_fft_bin_control_change, names='value')
update_fft_bin_status()

network_box = W.VBox([dst_ip, dst_mac, src_ip, src_mac, time_dst_port_base, spec_dst_port_base, time_src_port_base, spec_src_port_base, rust_receiver_url])
mode_box = W.VBox([science_output_mode, bandwidth_mhz, center_mhz, dac_mhz, W.HBox([fft_bin_align, align_dac_btn]), fft_bin_status, amplitude_percent, validation_seconds])
phase_grid = W.GridBox(phase_ch, layout=W.Layout(grid_template_columns='repeat(2, 270px)', grid_gap='4px 8px'))
phase_box = W.VBox([phase_grid, phase_btn])
preview_box = W.VBox([scope_y_codes, scope_display_points, waveform_view_mode, time_window_us, fft_padding_factor, preview_max_samples, refresh_hz, min_ui_sleep_ms, timeout_s, spectrum_floor_db, spectrum_top_db, spectrum_display_points, smoothing_enabled, smoothing_alpha])
controls = W.Tab(children=[network_box, mode_box, phase_box, preview_box])
for idx, title in enumerate(['Network', 'Science + DAC', '8-lane phase', 'Preview']):
    controls.set_title(idx, title)

button_row = W.HBox(
    [connect_btn, reload_btn, reprogram_pl, init_btn, apply_btn, capture_btn, start_live_btn, stop_live_btn, validate_btn, status_btn, stop_btn],
    layout=W.Layout(flex_flow='row wrap', align_items='center', grid_gap='6px 6px'),
)
status_tabs = W.Tab(children=[board_html, rust_html, preview_html])
for idx, title in enumerate(['Board', 'Rust receiver', 'Preview metrics']):
    status_tabs.set_title(idx, title)

display(HTML('<h2>Stage 27i TIME_SPEC 100MHz FFT-only Anti-alias Control + Preview</h2><p style="margin-top:-6px">Production contract: TIME ports 4300..4307, SPEC ports 4308..4323, 16 x 256-channel x 1-time FFT-only SPEC packets, 100MHz anti-alias active, CORE_VERSION 0x0001002B.</p>'))
display(button_row)
display(controls)
display(status_html)
display(W.VBox([rf_fig, spectrum_fig]))
display(status_tabs)
